# Data and feature audit

Read-only companion to Phases 1–2. The source of truth is `src/data/` and `src/features/`; this notebook displays their generated artifacts. Run `python -m src.data.prepare` and `python -m src.features.pipeline` first if artifacts are missing. It does not train or change data.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/project_contract.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open JupyterLab from the repository root (or its notebooks folder).")
print("Repository:", ROOT)


In [ ]:
quality = json.loads((ROOT / "reports/phase1_data_quality.json").read_text())
feature_quality = json.loads((ROOT / "reports/phase2_feature_quality.json").read_text())
manifest = json.loads((ROOT / "data/processed/phase2_feature_manifest.json").read_text())
labelled = pd.read_csv(ROOT / "data/processed/labelled_flights.csv", usecols=["sched_date", "label_status", "substantial_delay"], parse_dates=["sched_date"])
features = pd.read_csv(ROOT / "data/processed/phase2_features.csv", low_memory=False)
print("Source rows:", len(labelled))
print("Known labels:", labelled.label_status.eq("known").sum())
print("Unknown labels:", labelled.label_status.eq("unknown").sum())
print("Feature rows:", len(features), "Approved inputs:", len(manifest["allowlist"]))
assert len(features) == labelled.label_status.eq("known").sum()
assert features.flight_row_id.is_unique
assert not (set(manifest["allowlist"]) & set(manifest["denylist"]))


## Target and chronology

The target is `arr_delay >= 15` for known arrivals. Missing `arr_delay` means an unknown label and is excluded from the feature table.

In [ ]:
known = labelled.loc[labelled.label_status.eq("known")].copy()
monthly = known.groupby(known.sched_date.dt.month).substantial_delay.agg(flights="size", positive_rate="mean")
print(monthly.round(4).to_string())
ax = monthly.positive_rate.plot.bar(figsize=(9,4), title="Observed delay prevalence by scheduled month")
ax.set(xlabel="Month (2013)", ylabel="Fraction with arrival delay ≥15 minutes")
plt.tight_layout(); plt.show()


## Feature availability checks

The weather check verifies observation timestamps, not publication timestamps. Historical carrier and route rates use earlier **calendar dates**; the existing code does not verify that every prior arrival outcome was known by each T−60 cutoff.

In [ ]:
cutoff = pd.to_datetime(features.prediction_cutoff_utc, utc=True)
observed = pd.to_datetime(features.weather_observed_utc, utc=True)
future_weather = int((observed > cutoff).sum())
stale_used = int((features.weather_age_minutes.gt(180) & ~features.weather_stale_or_missing.astype(bool)).sum())
print("Weather observed after cutoff:", future_weather)
print("Stale weather values marked usable:", stale_used)
print("Stale or missing weather rows:", int(features.weather_stale_or_missing.sum()))
print("Carrier history missing rows:", int(features.carrier_history_missing.sum()))
print("Route history missing rows:", int(features.route_history_missing.sum()))
assert future_weather == 0 and stale_used == 0


## Interpretation

Review monthly prevalence shifts before interpreting ranking metrics. Retrospective weather observations are not archived forecasts. The historical-label availability question must be resolved in the pipeline before the final AML evaluation is treated as leakage-safe. See `reports/phase2_feature_quality.md` for coverage.